In [ ]:
import collections
import re
import tqdm
import pandas as pd
import os
import numpy as np
import astropy.units as u
import FunctionLib as FL
from pathlib import Path # Standard and correct way to import



In [ ]:
DJACatalog = FL.Spectrum_Catalog()
# Use a path with a tilde, which will now be correctly expanded
my_directory = '~/DJAv4.2' # Or the full path to your unzipped directory
all_spec_files = DJACatalog.load_spectrum_filepaths_from_directory(my_directory)

print(f"Found {len(all_spec_files)} spec.fits files in the directory {my_directory}.")

In [ ]:
import pandas as pd
from pathlib import Path
from typing import List

def find_missing_files(
    df: pd.DataFrame,
    base_directory: str,
    column_name: str = 'file'
) -> List[str]:
    """
    Compares file paths in a DataFrame column against a base directory to find which files are missing.

    Args:
        df (pd.DataFrame): The DataFrame containing the list of expected files.
        base_directory (str): The root directory where files were supposed to be downloaded.
                                Tilde (~) for the home directory is supported.
        column_name (str): The name of the column in the DataFrame that contains the relative file paths.
                           Defaults to 'file'.

    Returns:
        List[str]: A list of relative file paths from the DataFrame that were not found on the filesystem.
    """
    # 1. Expand the tilde (~) and create a Path object for the base directory.
    base_path = Path(base_directory).expanduser()

    # Check if the base directory actually exists.
    if not base_path.is_dir():
        print(f"Error: The base directory '{base_path}' does not exist.")
        # Return all file paths as missing if the root directory is not there.
        return df[column_name].tolist()

    missing_files = []

    # 2. Check for the existence of the specified column.
    if column_name not in df.columns:
        print(f"Error: Column '{column_name}' not found in the DataFrame.")
        return []

    print(f"Checking {len(df)} file paths against the directory '{base_path}'...")

    # 3. Iterate through each file path in the DataFrame column.
    for relative_path in df[column_name]:
        # 4. Combine the base path and the relative path to get the full path.
        full_path = base_path / relative_path

        # 5. Check if the file does not exist.
        if not full_path.is_file():
            # 6. If missing, add it to our list.
            missing_files.append(relative_path)

    return missing_files


your_download_directory = '~/DJAv4.2'
df = pd.read_csv('./DJAv4.2Catalog.csv')
all_spec_filenames = {os.path.basename(f) for f in all_spec_files}
missing_file_list = find_missing_files(df, your_download_directory)


# 2. 提取DataFrame中的'file'列，并转换为一个集合(set)以提高比较效率
data_files = set(df['file'])

# 3. 将您的列表也转换为一个集合
all_spec_files_set = set(all_spec_filenames)
all_spec_filenames_set = {os.path.basename(f) for f in all_spec_filenames}

# 3. 使用pandas的 isin() 方法来筛选DataFrame
# isin() 会检查 'file' 列中的每个值是否存在于 all_spec_filenames_set 中
# 我们在前面加上 '~' 符号，表示取反，即筛选出那些 *不* 存在于集合中的行
missing_rows_df = df[~df['file'].isin(all_spec_filenames_set)]

# 4. 检查是否有找到缺失的条目，并将它们写入新的CSV文件
if not missing_rows_df.empty:
    # 将包含所有缺失条目的新DataFrame写入到CSV文件中
    # index=False 表示在写入文件时不包含DataFrame的行索引号
    missing_rows_df.to_csv('missing_entries.csv', index=False)

    print(f"检查完成。共找到 {len(missing_rows_df)} 条缺失的记录。")
    print("这些完整的条目已经写入到 'missing_entries.csv' 文件中。")
else:
    print("检查完成。您CSV文件中的所有条目都在 all_spec_files 列表中找到了对应文件。")

In [ ]:
len(all_spec_files)

In [ ]:
DJACatalog=FL.Spectrum_Catalog()
DJACatalog.process_files(all_spec_files, DJA_Catalog_DataFrame=df)

In [ ]:
DJACatalog.to_dataframe()

In [ ]:
DJACatalog.save_catalog_to_pkl('DJAV4.2Catalog.pkl')

In [6]:
Catalog=FL.Spectrum_Catalog()
Catalog.load_from_pkl('DJAV4.2Catalog.pkl')

In [7]:
Catalog.to_dataframe()

,survey_id_subid,prism_filepath,prism_redshift,determined_redshift,grating_filepaths,grating_redshifts,file_count,available_filters,properties,survey_id,sample_flag,reason_for_exclusion,grating_slitloss_correction,grating_within_coverage,lines_fit,s3fit_result_path,line_ratios,dust_curve_parameters
0,snh0pe-v4_4446_102,/home/xingyaocai/DJAv4.2/snh0pe-v4/snh0pe-v4_p...,0.2259,0.2259,{'g140m-f100lp': '/home/xingyaocai/DJAv4.2/snh...,"{'g140m-f100lp': nan, 'g235m-f170lp': nan}",3,"{g140m-f100lp, g235m-f170lp, prism-clear}",{'redshift_conflict': False},snh0pe-v4,False,"[redshift_below_3, redshift_below_3, redshift_...",{},{},{},None,{},{}
1,snh0pe-v4_4446_143,/home/xingyaocai/DJAv4.2/snh0pe-v4/snh0pe-v4_p...,1.6318,1.6311,{'g140m-f100lp': '/home/xingyaocai/DJAv4.2/snh...,"{'g140m-f100lp': 1.6313, 'g235m-f170lp': 1.6309}",3,"{g140m-f100lp, g235m-f170lp, prism-clear}",{'redshift_conflict': False},snh0pe-v4,False,"[redshift_below_3, redshift_below_3, redshift_...",{},{},{},None,{},{}
2,snh0pe-v4_4446_285,/home/xingyaocai/DJAv4.2/snh0pe-v4/snh0pe-v4_p...,0.4446,0.4462,{'g235m-f170lp': '/home/xingyaocai/DJAv4.2/snh...,"{'g235m-f170lp': 0.4462, 'g140m-f100lp': 0.4462}",3,"{g140m-f100lp, g235m-f170lp, prism-clear}",{'redshift_conflict': False},snh0pe-v4,False,"[redshift_below_3, redshift_below_3, redshift_...",{},{},{},None,{},{}
3,snh0pe-v4_4446_29,/home/xingyaocai/DJAv4.2/snh0pe-v4/snh0pe-v4_p...,1.7834,1.77975,{'g140m-f100lp': '/home/xingyaocai/DJAv4.2/snh...,"{'g140m-f100lp': 1.7799, 'g235m-f170lp': 1.7796}",3,"{g140m-f100lp, g235m-f170lp, prism-clear}",{'redshift_conflict': False},snh0pe-v4,False,"[redshift_below_3, redshift_below_3, redshift_...",{},{},{},None,{},{}
4,snh0pe-v4_4446_123,/home/xingyaocai/DJAv4.2/snh0pe-v4/snh0pe-v4_p...,1.7855,1.7855,{'g140m-f100lp': '/home/xingyaocai/DJAv4.2/snh...,"{'g140m-f100lp': 1.7855, 'g235m-f170lp': 1.7851}",3,"{g140m-f100lp, g235m-f170lp, prism-clear}",{'redshift_conflict': False},snh0pe-v4,False,"[redshift_below_3, redshift_below_3, redshift_...",{},{},{},None,{},{}
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43037,glimpse-obs02-v4_9223_9152,None,None,5.5385,{'g395m-f290lp': '/home/xingyaocai/DJAv4.2/gli...,{'g395m-f290lp': 5.5385},1,{g395m-f290lp},{'redshift_conflict': False},glimpse-obs02-v4,False,"[no_prism_spectrum, no_prism_spectrum, no_pris...",{},{},{},None,{},{}
43038,glimpse-obs02-v4_9223_98001,None,None,2.6322,{'g395m-f290lp': '/home/xingyaocai/DJAv4.2/gli...,{'g395m-f290lp': 2.6322},1,{g395m-f290lp},{'redshift_conflict': False},glimpse-obs02-v4,False,"[no_prism_spectrum, no_prism_spectrum, no_pris...",{},{},{},None,{},{}
43039,glimpse-obs02-v4_9223_47046,None,None,4.3554,{'g395m-f290lp': '/home/xingyaocai/DJAv4.2/gli...,{'g395m-f290lp': 4.3554},1,{g395m-f290lp},{'redshift_conflict': False},glimpse-obs02-v4,False,"[no_prism_spectrum, no_prism_spectrum, no_pris...",{},{},{},None,{},{}
43040,glimpse-obs02-v4_9223_45350,None,None,1.3675,{'g395m-f290lp': '/home/xingyaocai/DJAv4.2/gli...,{'g395m-f290lp': 1.3675},1,{g395m-f290lp},{'redshift_conflict': False},glimpse-obs02-v4,False,"[no_prism_spectrum, no_prism_spectrum, no_pris...",{},{},{},None,{},{}


In [8]:
Catalog.get_summary_stats()

{'total_objects': 43042,
 'with_prism': 34949,
 'without_prism': 8093,
 'with_grating': 21770,
 'total_spectra': 80367,
 'total_prism_spectra': 34949,
 'total_grating_spectra': 45418}